# PlantCare AI — Notebook 01 v6
### Classification only · Grape + Tomato · preprocessing on all splits · rebuilt lesion segmentation

This notebook does one job: **train and evaluate the disease classifier**, and produce the
lesion mask and severity that go with each prediction. No LLM, no retrieval, no advisory
text — that belongs in its own notebook, and mixing it in here means a change to a prompt
forces you to re-run training.

**What changed against v4.**

| # | Change | Where |
|---|---|---|
| 1 | **LLM layer removed entirely.** Section 11 and every advisory field are gone | — |
| 2 | **Preprocessing now runs on train, val AND test** | sections 2, 4, 7 |
| 3 | **A real split-leakage audit**, since that is the thing that actually leaks | section 3c |
| 4 | Lesion segmentation rebuilt (carried over from v4) | section 5 |

---

### On preprocessing and leakage

Applying blur removal and brightness correction to validation and test is **not** leakage.
Leakage means information crossing between splits. These two operations are per-image and
self-contained: each image's own Laplacian variance and its own mean grey level decide what
happens to that image, and no statistic is pooled across the dataset. That makes them a
fixed input transform, like resizing — not a fitted one like a `StandardScaler`.

Applying them to train only would have been the riskier choice: the model would learn on
sharpened, brightness-corrected images and then meet raw ones, which is a train/serve
mismatch.

One rule still applies, and section 4b enforces it by construction: `BLUR_VARIANCE_MIN` and
the brightness band are tuned on the **training split only**, and never re-tuned after
looking at test results. Tuning a threshold against the test set is how a per-image
transform becomes a fitted one.

### The leakage that is actually worth worrying about

Your dataset has no split folders, so the notebook carves val and test out of train at
random. Plant-disease datasets are full of **multiple photographs of the same physical
leaf**, and augmented copies of one source image. If two shots of one leaf land on
opposite sides of the split, test accuracy measures memorisation, not generalisation — and
that inflates the score far more than any preprocessing choice could.

Section 3c hashes every image (exact MD5 plus a 64-bit perceptual dHash), finds groups that
straddle splits, and with `FIX_SPLIT_LEAKAGE = True` moves each whole group into a single
split before training. Run it and read the number it prints; if it is large, your previous
accuracy figures were optimistic.

### Two things to be honest about

*Accuracy is not comparable to the 27-class runs.* Grape and tomato were already at
0.99–1.00 F1 there; wheat carried every error. Expect ~0.99 and always report the class
count with it.

*Severity bands need calibrating.* The rebuilt mask counts necrotic tissue that the old one
deleted, so the same leaf now reads higher. `HEALTHY_SEVERITY` / `MILD_MAX` /
`MODERATE_MAX` are an estimate until you check them against your own histogram in section 6.

## 1. Install + imports

In [ ]:
!pip install -q -U openpyxl scikit-learn

from pathlib import Path
import inspect
import json
import math
import os
import random
import re
import shutil
import textwrap
import warnings
from collections import defaultdict

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms

from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Settings

`CROPS` lists the folder names the path parser should read as a *crop* level rather than a
*class* level — grape and tomato, which is everything your dataset contains. The
preprocessing block and the leakage-audit block are the two you are most likely to change.

In [ ]:
# ---------------- where to look / where to write ----------------
DATASET_ROOT_OVERRIDE = None                 # set a path string here to skip auto-detection
SEARCH_ROOTS = ["/kaggle/input", "/content", "./data", "."]   # scanned in this order

PROJECT_ROOT = Path("/kaggle/working/PlantCare_CNN")

MANIFEST_PATH   = PROJECT_ROOT / "manifest.csv"
CLASS_INDEX     = PROJECT_ROOT / "class_index.json"
CHECKPOINT_PATH = PROJECT_ROOT / "best_cnn.pt"
PREVIEW_ROOT    = PROJECT_ROOT / "lesion_previews"
PREDICTION_ROOT = PROJECT_ROOT / "final_predictions"
EXPORT_ROOT     = PROJECT_ROOT / "artifacts"

for folder in [PROJECT_ROOT, PREVIEW_ROOT, PREDICTION_ROOT, EXPORT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

# ---------------- how folders are read ----------------
# Crop names the path parser recognises as a crop level rather than a class level.
CROPS = {"grape", "tomato"}
HEALTHY_NAMES = {"healthy", "health", "normal", "fresh", "nodisease"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
SPLIT_MAPPING = {"train": "train", "training": "train",
                 "validation": "val", "val": "val", "valid": "val",
                 "test": "test", "testing": "test"}

VAL_FRACTION  = 0.15    # carved out of train only if no validation folder exists
TEST_FRACTION = 0.15    # carved out of train only if no test folder exists

# ---------------- preprocessing, and which splits get it ----------------
# ALL THREE SPLITS. Blur removal and brightness correction are per-image operations —
# each image's own Laplacian variance and its own mean grey level decide what happens to
# it, and nothing is pooled across images. So this is a fixed input transform like
# resizing, not a fitted one, and applying it everywhere carries no leakage. Applying it
# to train only would instead create a train/serve mismatch.
#
# The rule that DOES matter: the constants below are tuned on the TRAINING split only,
# in section 4b. Never re-tune them after looking at test results.
PREPROCESS_SPLITS = {"train", "val", "test"}

# ---------------- split-leakage audit ----------------
RUN_LEAKAGE_AUDIT = True     # hashes every image; a few minutes on 40k files
FIX_SPLIT_LEAKAGE = True     # move duplicate groups so they sit in exactly one split
DHASH_SIZE        = 8        # 64-bit perceptual hash
NEAR_DUP_MAX_BITS = 3        # Hamming distance at or below this = same photo

BLUR_VARIANCE_MIN = 120.0   # Laplacian variance below this -> image is blurry, sharpen it
UNSHARP_RADIUS    = 3.0     # Gaussian sigma of the unsharp mask
UNSHARP_AMOUNT    = 0.8     # strength; >1.5 starts producing halos on leaf veins
BRIGHTNESS_TARGET = 128.0   # mid-grey
BRIGHTNESS_LOW    = 95.0    # mean grey below this -> too dark
BRIGHTNESS_HIGH   = 165.0   # mean grey above this -> too bright
GAMMA_LIMITS      = (0.55, 1.85)   # never push gamma further than this
CLAHE_CLIP        = 2.0
CLAHE_GRID        = 8

# ---------------- lesion segmentation (rebuilt — see section 5) ----------------
WORK_SIZE          = 512     # long side used for mask maths (speed)

# STAGE 1 — leaf vs background, by colour distance from a border-sampled background.
# Distance uses the Lab a/b (chroma) channels ONLY. Lightness is excluded deliberately:
# on this dataset the background's chroma MAD is ~2 while its lightness MAD is ~28
# because of the cast shadow, so any lightness weighting drags the shadow into the leaf
# and then scores it as lesion.
BORDER_FRACTION    = 0.05    # width of the border strip sampled as "background"
CHROMA_FLOOR       = 6.0     # minimum Lab a/b distance to count as "not background"
BG_UNIFORM_MAD     = 12.0    # border chroma MAD above this -> not a uniform background,
                             #   fall back to the excess-green mask and flag the image
LEAF_CLOSE_RATIO   = 0.02    # keep SMALL: a big close fills the gaps between grape lobes
                             #   and every background pixel it traps becomes a "lesion"

# near-black necrosis has no chroma, so stage 1 cannot see it; rescued by enclosure
DARK_DROP          = 40.0    # L this far below the background counts as "dark"
DARK_ENCLOSURE     = 0.55    # fraction of the blob's border that must touch leaf tissue
DARK_MIN_AREA      = 40

# STAGE 2 — healthy vs diseased, INSIDE the leaf only. Everything in the leaf that is
# not green-and-saturated is a lesion.
HUE_LOW, HUE_HIGH  = 25, 95  # OpenCV hue range for leaf green
SAT_MIN            = 45
LESION_SMOOTH_RATIO = 0.008

MIN_LESION_RATIO   = 0.0008  # blob must be >= this fraction of leaf area to earn a box
MAX_LESION_RATIO   = 0.60    # a single blob may now legitimately be most of the leaf
MAX_BOXES          = 20
BOX_PADDING_RATIO  = 0.01

# leaf-mask sanity gate: a "leaf" filling almost nothing or almost the whole frame means
# the mask failed, so severity is reported as "Unavailable" rather than a wrong number.
MIN_LEAF_FRACTION  = 0.04
MAX_LEAF_FRACTION  = 0.96

# ---------------- severity grading ----------------
# These bands are LARGER than the old notebook's because the leaf mask changed: the
# denominator now includes necrotic tissue, so the same leaf reads higher. Section 6b
# recomputes suggested bands from your own healthy/diseased distributions — run it.
HEALTHY_SEVERITY  = 5.0
MILD_MAX          = 15.0
MODERATE_MAX      = 35.0

# ---------------- CNN settings ----------------
BACKBONE      = "efficientnet_b0"   # only backbone supported in this notebook
IMAGE_SIZE    = 224
BATCH_SIZE    = 64
EPOCHS        = 15
LEARNING_RATE = 3e-4
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 2
USE_LESION_CHANNEL = False   # True -> feed a 4th channel (the lesion mask) into the CNN
BALANCE_CLASSES    = True
TRAIN_MODEL        = True
MAX_IMAGES_PER_CLASS = None  # e.g. 500 for a quick smoke test, None for everything
CONFIDENCE_GATE    = 0.80    # below this -> refuse, return the white "cannot classify" card

# ---------------- rejecting inputs the model has no business classifying ----------------
# A 13-class softmax ALWAYS sums to 1, so an unseen leaf type does not produce low
# confidence — it produces a confident wrong answer. Softmax alone cannot detect it.
# Three independent screens, section 11:
#   1. is there a leaf in the frame at all      (reuses the section 5 mask)
#   2. does the image look like the training set (Mahalanobis distance in feature space)
#   3. is the classifier confident               (the 0.80 gate above)
VEGETATION_MIN     = 0.06    # leaf must cover at least this much of the frame
VEGETATION_MAX     = 0.97    # ...and not effectively all of it
OOD_RETENTION      = 0.995   # keep this share of VALIDATION images; the rest sets the
                             #   threshold. Lower it to reject harder, at the cost of
                             #   refusing more genuine grape/tomato photos.
FEATURE_SHRINKAGE  = 0.01    # ridge added to the covariance before inversion
EXTRA_OOD_DIR      = None    # optional folder of known out-of-distribution images to
                             #   score in section 11c (e.g. your old wheat folder)
REJECT_CARD_SIZE   = 640     # px, the white "cannot classify" image

print("Project    :", PROJECT_ROOT)
print("Crops      :", sorted(CROPS))
print("Preprocess :", sorted(PREPROCESS_SPLITS))

## 2b. Find the dataset by itself

Kaggle slugs change and datasets get re-uploaded with an extra folder wrapped around them,
so instead of naming a path this section **looks for one**: it walks the search roots,
records every directory that actually contains image files, and takes the deepest folder
containing all of them.

On your tree that lands on `.../CROP_Dataset`, because `Grape/` and `Tomato/` are its
children. Read the printed path and image count before running anything else — every cell
below inherits it.

In [ ]:
def scan_folders(root, exts):
    """{folder: file count} for every folder under root holding files with these extensions."""
    found = {}
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        count = sum(1 for f in filenames if Path(f).suffix.lower() in exts)
        if count:
            found[Path(dirpath)] = count
    return found


def _group_key(folder, search_root):
    """Top-level dataset folder that `folder` sits in — used to separate attached datasets."""
    relative = folder.relative_to(search_root)
    return search_root / relative.parts[0] if relative.parts else search_root


def detect_dataset_root(search_roots=SEARCH_ROOTS, exts=IMAGE_EXTENSIONS):
    """Deepest folder containing every image folder of the largest dataset found."""
    groups, counts = defaultdict(list), defaultdict(int)

    for candidate in search_roots:
        search_root = Path(candidate).resolve()
        if not search_root.is_dir():
            continue
        for folder, count in scan_folders(search_root, exts).items():
            key = _group_key(folder, search_root)
            groups[key].append(folder)
            counts[key] += count
        if groups:            # first search root that yielded anything wins
            break

    if not groups:
        raise FileNotFoundError(
            "No image folders found under: " + ", ".join(map(str, search_roots)) +
            "\nAttach the dataset, or set DATASET_ROOT_OVERRIDE."
        )

    best = max(counts, key=counts.get)
    folders = groups[best]
    root = Path(os.path.commonpath([str(f) for f in folders]))
    return root, folders, counts[best]


def print_tree(root, max_depth=3, max_children=12):
    print(root.name + "/")
    for folder in sorted(p for p in root.rglob("*") if p.is_dir()):
        depth = len(folder.relative_to(root).parts)
        if depth > max_depth:
            continue
        siblings = sorted(p for p in folder.parent.iterdir() if p.is_dir())
        if siblings.index(folder) >= max_children:
            continue
        images = sum(1 for f in folder.iterdir()
                     if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS)
        tail = f"   [{images} images]" if images else ""
        print("    " * depth + folder.name + "/" + tail)


if DATASET_ROOT_OVERRIDE:
    DATASET_ROOT = Path(DATASET_ROOT_OVERRIDE)
    scanned = scan_folders(DATASET_ROOT, IMAGE_EXTENSIONS)
    image_folders = sorted(scanned)
    total_images = sum(scanned.values())
    print("Using DATASET_ROOT_OVERRIDE")
else:
    DATASET_ROOT, image_folders, total_images = detect_dataset_root()

# note where the RAG PDFs live so notebook 02 can find them; nothing here reads them
pdf_folders = {}
for candidate in {DATASET_ROOT, *(p for p in DATASET_ROOT.parents)}:
    if str(candidate) in ("/", "/kaggle", "/kaggle/input"):
        continue
    pdf_folders.update(scan_folders(candidate, {".pdf"}))
RAG_DOC_ROOT = sorted(pdf_folders, key=pdf_folders.get, reverse=True)[0] if pdf_folders else None

print("\nDATASET_ROOT :", DATASET_ROOT)
print("RAG_DOC_ROOT :", RAG_DOC_ROOT)
print(f"{total_images} images across {len(image_folders)} folders\n")
print_tree(DATASET_ROOT)

## 3. Extract the class of every image — first, before anything else

No copying, no `labels/` folder: one CSV manifest, built by reading the path of each image
folder. Crop, split and class are recovered from the path parts, so the loop does not care
whether the layout is `Grape/train/Black_rot`, `train/Grape/Black_rot`, or — as in your
dataset — has no split level at all. Missing splits get carved in the next cell.

In [ ]:
def clean_name(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")


def is_healthy_class(name):
    return bool(HEALTHY_NAMES & set(clean_name(name).split("_")))


records = []

for folder in sorted(image_folders):
    paths = sorted(f for f in folder.iterdir()
                   if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS)
    if not paths:
        continue

    parts = [clean_name(part) for part in folder.relative_to(DATASET_ROOT).parts]

    split = next((SPLIT_MAPPING[p] for p in parts if p in SPLIT_MAPPING), "train")
    crop  = next((p for p in parts if p in CROPS), None)
    label = next((p for p in reversed(parts)
                  if p not in SPLIT_MAPPING and p not in CROPS), parts[-1])
    crop  = crop or label.split("_")[0]

    class_name = label if label.startswith(crop) else f"{crop}_{label}"

    if MAX_IMAGES_PER_CLASS:
        paths = paths[:MAX_IMAGES_PER_CLASS]

    for p in paths:
        records.append({
            "filepath": str(p),
            "crop": crop,
            "raw_class": folder.name,
            "class_name": class_name,
            "split": split,
            "healthy": is_healthy_class(class_name),
        })

manifest = pd.DataFrame(records)
if manifest.empty:
    raise ValueError("Manifest is empty — check the tree printed above and DATASET_ROOT.")

class_names = sorted(manifest["class_name"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
id_to_class = {i: name for name, i in class_to_id.items()}
manifest["label_id"] = manifest["class_name"].map(class_to_id)

NUM_CLASSES = len(class_names)
print(f"{len(manifest)} images | {NUM_CLASSES} classes | crops: {sorted(manifest['crop'].unique())}")
print(manifest["split"].value_counts().to_string())
display(manifest.groupby(["split", "class_name"]).size().unstack(fill_value=0).T)

Splits are only carved if the dataset does not already provide them. Both carves are
stratified by class and come out of `train`, so an existing `validation/` or `test/`
folder is left exactly as the dataset authors made it.

In [ ]:
def carve(frame, wanted, fraction):
    """Move a stratified slice of train into `wanted`. Returns the modified frame."""
    train_part = frame[frame["split"] == "train"]
    counts = train_part["label_id"].value_counts()

    # a stratified slice needs at least one image per class in it
    size = max(int(round(len(train_part) * fraction)), len(counts))
    stratify = train_part["label_id"] if counts.min() >= 2 else None

    try:
        _, moved = train_test_split(train_part.index, test_size=size,
                                    stratify=stratify, random_state=SEED)
    except ValueError:                      # too few images per class to stratify
        _, moved = train_test_split(train_part.index, test_size=size, random_state=SEED)

    frame.loc[moved, "split"] = wanted
    print(f"carved {len(moved)} images out of train -> {wanted}")
    return frame


present = set(manifest["split"])
if "val" not in present:
    manifest = carve(manifest, "val", VAL_FRACTION)
if "test" not in present:
    manifest = carve(manifest, "test", TEST_FRACTION)

manifest.to_csv(MANIFEST_PATH, index=False)
CLASS_INDEX.write_text(json.dumps(class_to_id, indent=2), encoding="utf-8")

print(manifest["split"].value_counts().to_string())
print("manifest ->", MANIFEST_PATH)

## 3b. Read the manifest before you train on it

The class list here is what gets baked into `class_index.json` and then into the
checkpoint, and a class list that does not match the one your Flask app loads is a bug you
only discover at deployment. Check the names and the count against what you expect.

The imbalance ratio decides whether `BALANCE_CLASSES` is earning its place: under about
3x the weighted sampler is mostly adding variance for nothing.

In [ ]:
print(f"{NUM_CLASSES} classes across {len(manifest)} images\n")
for crop in sorted(manifest["crop"].unique()):
    members = [c for c in class_names if c.startswith(crop)]
    print(f"  {crop:8s} {len(members):2d}  {[m.replace(crop + '_', '') for m in members]}")

train_counts = manifest[manifest["split"] == "train"]["class_name"].value_counts()
imbalance = train_counts.max() / max(1, train_counts.min())
print(f"\ntrain imbalance {imbalance:.1f}x "
      f"(max {train_counts.idxmax()} {train_counts.max()}, "
      f"min {train_counts.idxmin()} {train_counts.min()})")
if imbalance < 3.0 and BALANCE_CLASSES:
    print("  -> under 3x; BALANCE_CLASSES is probably adding variance for little gain")

# every class must survive into every split, or the metrics silently lie
missing = (manifest.groupby(["split", "class_name"]).size()
                   .unstack(fill_value=0).eq(0).any(axis=0))
assert not missing.any(), f"classes absent from at least one split: {list(missing[missing].index)}"
print("every class is present in train, val and test")

## 3c. Split-leakage audit — the leak that actually matters

Your dataset ships no `train/val/test` folders, so section 3 carved them at random. Random
carving is fine when every image is an independent observation. It is not fine when the
dataset contains several photographs of the **same physical leaf**, or augmented copies of
one source image — and plant-disease datasets usually do.

If two shots of one leaf land on opposite sides of the split, the model can memorise that
leaf and score on it at test time. That is real leakage, and it inflates accuracy far more
than any preprocessing decision.

Two detectors, because they catch different things:

* **MD5 of the file bytes** — catches exact duplicates: the same file copied into two
  folders. Cheap and certain.
* **Perceptual dHash** — a 64-bit signature of the brightness gradient in an 8x9 thumbnail.
  Two images within `NEAR_DUP_MAX_BITS` of each other are the same photo after a resize,
  a re-compression, or a small crop. This is the one that catches augmented copies.

With `FIX_SPLIT_LEAKAGE = True` every duplicate group is moved into a single split, chosen
by majority. The fix runs **before** the loaders are built, so nothing downstream sees the
leaked arrangement. Read the number it prints: if it is not zero, your earlier accuracy
figures were optimistic and this run is the honest one.

In [ ]:
import hashlib


def file_md5(path, chunk=1 << 20):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def dhash(path, size=DHASH_SIZE):
    """64-bit perceptual hash: which way does brightness step, left to right."""
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return None
    small = cv2.resize(image, (size + 1, size), interpolation=cv2.INTER_AREA)
    bits = small[:, 1:] > small[:, :-1]
    value = 0
    for bit in bits.flatten():
        value = (value << 1) | int(bit)
    return value


def popcount(value):
    return bin(value).count("1")


def group_near_duplicates(hashes, max_bits=NEAR_DUP_MAX_BITS):
    """Bucket by 16-bit prefix first, then compare within buckets — O(n) in practice."""
    buckets = defaultdict(list)
    for index, value in hashes.items():
        for shift in (48, 32, 16, 0):                 # 4 bands; near-dupes share one
            buckets[(shift, (value >> shift) & 0xFFFF)].append(index)

    parent = {i: i for i in hashes}

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for members in buckets.values():
        if len(members) < 2 or len(members) > 400:    # skip degenerate buckets
            continue
        for a_i in range(len(members)):
            for b_i in range(a_i + 1, len(members)):
                a, b = members[a_i], members[b_i]
                if find(a) != find(b) and popcount(hashes[a] ^ hashes[b]) <= max_bits:
                    union(a, b)

    groups = defaultdict(list)
    for i in hashes:
        groups[find(i)].append(i)
    return [g for g in groups.values() if len(g) > 1]


if RUN_LEAKAGE_AUDIT:
    print(f"hashing {len(manifest)} images — this takes a few minutes")
    manifest["md5"] = [file_md5(p) for p in manifest["filepath"]]
    hashes = {}
    for i, path in zip(manifest.index, manifest["filepath"]):
        value = dhash(path)
        if value is not None:
            hashes[i] = value

    exact = [g.tolist() for _, g in manifest.groupby("md5").groups.items()
             if len(g) > 1]
    near = group_near_duplicates(hashes)

    def straddles(group):
        return manifest.loc[group, "split"].nunique() > 1

    exact_cross = [g for g in exact if straddles(g)]
    near_cross = [g for g in near if straddles(g)]

    print(f"\nexact duplicate groups (md5)      : {len(exact):5d}  "
          f"of which cross a split: {len(exact_cross)}")
    print(f"near-duplicate groups (dhash<={NEAR_DUP_MAX_BITS})  : {len(near):5d}  "
          f"of which cross a split: {len(near_cross)}")
    leaked_images = sum(len(g) for g in near_cross)
    print(f"images involved in a cross-split group: {leaked_images} "
          f"({leaked_images / len(manifest):.2%} of the dataset)")

    if near_cross:
        print("\nexamples:")
        for group in near_cross[:3]:
            for i in group[:3]:
                row = manifest.loc[i]
                print(f"   [{row['split']:5s}] {row['class_name']:28s} {Path(row['filepath']).name}")
            print("   ---")

    if FIX_SPLIT_LEAKAGE and near_cross:
        moved = 0
        for group in near_cross:
            winner = manifest.loc[group, "split"].value_counts().idxmax()
            for i in group:
                if manifest.at[i, "split"] != winner:
                    manifest.at[i, "split"] = winner
                    moved += 1
        manifest.to_csv(MANIFEST_PATH, index=False)
        print(f"\nmoved {moved} images so every duplicate group sits in one split")
        print(manifest["split"].value_counts().to_string())
    elif near_cross:
        print("\nFIX_SPLIT_LEAKAGE is False — the leak is reported but NOT repaired.")
    else:
        print("\nno cross-split duplicates: the random carve is safe on this dataset")
else:
    print("leakage audit skipped (RUN_LEAKAGE_AUDIT = False)")

## 4. Preprocessing — blur removal and brightness correction

**Both operations are conditional.** A sharp, well-lit image is returned byte-identical:
sharpening an already-sharp leaf adds halos along the veins, and CLAHE on a correctly
exposed image lifts background texture into something the mask mistakes for a lesion. The
gates are what keep preprocessing from becoming damage.

*Blur.* `cv2.Laplacian(...).var()` measures how much high-frequency detail survives. A
crisp leaf scores in the hundreds; a soft phone photo scores in the tens. Below
`BLUR_VARIANCE_MIN` an unsharp mask fires — `original + amount * (original − blurred)`,
which re-weights the edges the blur flattened. It cannot recover detail that was never
recorded; it makes the surviving edges legible.

*Brightness.* Two stages, because they fix different faults. CLAHE on the **L channel of
LAB** fixes *local* contrast — the shadowed half of a leaf — while leaving A and B alone so
the colour never shifts, which matters because the lesion mask downstream is a colour
rule. A gamma correction toward mid-grey then fixes *global* exposure, clamped by
`GAMMA_LIMITS` so a genuinely dark image is lifted rather than blown out.

Order is deliberate: sharpen, then brighten. Brightening first would amplify the noise the
sharpener then multiplies again.

In [ ]:
def blur_score(bgr):
    """Variance of the Laplacian. High = sharp, low = blurry. Scale-dependent, so it is
    only meaningful compared against other images of similar size."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def brightness_score(bgr):
    """Mean grey level, 0-255. ~128 is well exposed."""
    return float(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).mean())


def remove_blur(bgr, radius=UNSHARP_RADIUS, amount=UNSHARP_AMOUNT):
    """Unsharp mask: add back a scaled copy of what the blur removed."""
    blurred = cv2.GaussianBlur(bgr, (0, 0), radius)
    return cv2.addWeighted(bgr, 1.0 + amount, blurred, -amount, 0)


def correct_brightness(bgr, target=BRIGHTNESS_TARGET, clip=CLAHE_CLIP, grid=CLAHE_GRID,
                       gamma_limits=GAMMA_LIMITS):
    """CLAHE on LAB-L for local contrast, then a clamped gamma toward mid-grey.

    A and B are untouched on purpose — the lesion mask is a colour rule, so shifting hue
    here would move every severity number downstream."""
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(int(grid), int(grid))).apply(l)
    out = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

    mean = brightness_score(out)
    if 1.0 < mean < 254.0:
        # solve (mean/255) ** (1/gamma) == target/255  for gamma.
        # gamma > 1 brightens, gamma < 1 darkens. Getting this ratio the wrong way
        # round silently pushes dark images darker, so it is asserted below.
        gamma = math.log(mean / 255.0) / math.log(target / 255.0)
        gamma = float(np.clip(gamma, *gamma_limits))
        if abs(gamma - 1.0) > 0.02:
            table = np.array([((i / 255.0) ** (1.0 / gamma)) * 255
                              for i in range(256)], dtype=np.uint8)
            out = cv2.LUT(out, table)
    return out


def preprocess_image(bgr, blur_min=BLUR_VARIANCE_MIN,
                     bright_low=BRIGHTNESS_LOW, bright_high=BRIGHTNESS_HIGH,
                     report=False):
    """Conditional clean-up. Returns the image, or (image, report) when report=True.

    A good image passes through untouched — check `deblurred` / `rebalanced` in the
    report to see whether anything actually fired."""
    info = {
        "blur_before": blur_score(bgr),
        "brightness_before": brightness_score(bgr),
        "deblurred": False,
        "rebalanced": False,
    }

    out = bgr
    if info["blur_before"] < blur_min:
        out = remove_blur(out)
        info["deblurred"] = True

    if not (bright_low <= info["brightness_before"] <= bright_high):
        out = correct_brightness(out)
        info["rebalanced"] = True

    if report:
        info["blur_after"] = blur_score(out)
        info["brightness_after"] = brightness_score(out)
        info["changed"] = info["deblurred"] or info["rebalanced"]
        return out, info
    return out


# direction test — a dark image must come out brighter, a bright one darker.
# This is here because an inverted gamma is silent: it produces a plausible image
# and quietly degrades every training sample the gate touches.
_dark = np.full((64, 64, 3), 45, np.uint8)
_bright = np.full((64, 64, 3), 215, np.uint8)
assert brightness_score(correct_brightness(_dark)) > brightness_score(_dark), "gamma inverted"
assert brightness_score(correct_brightness(_bright)) < brightness_score(_bright), "gamma inverted"
print("brightness direction OK:",
      f"{brightness_score(_dark):.0f}->{brightness_score(correct_brightness(_dark)):.0f}",
      f"| {brightness_score(_bright):.0f}->{brightness_score(correct_brightness(_bright)):.0f}")

_probe_bgr = cv2.imread(manifest["filepath"].iloc[0])
_, _probe_report = preprocess_image(_probe_bgr, report=True)
print(manifest["class_name"].iloc[0], "->", {k: (round(v, 1) if isinstance(v, float) else v)
                                             for k, v in _probe_report.items()})

## 4b. Calibrate the gates before you trust them

Two questions, and you should answer both before training on cleaned data.

**How often does each gate fire?** If deblurring fires on 2% of the training set it is
doing nothing; if it fires on 95% the threshold is wrong and you are sharpening the whole
dataset unconditionally. Somewhere in the 5–40% band is where the gate is behaving like a
gate. Tune `BLUR_VARIANCE_MIN` until it lands there.

**Does the output actually look better?** Firing rates cannot tell you that. Look at the
before/after pairs — you are checking for halo rings on the veins and for washed-out
lesion edges, which are the two ways this helps the metric and hurts the model.

In [ ]:
train_frame = manifest[manifest["split"] == "train"]
calib = train_frame.sample(min(400, len(train_frame)), random_state=SEED)

rows = []
for _, row in calib.iterrows():
    bgr = cv2.imread(row["filepath"])
    if bgr is None:
        continue
    _, info = preprocess_image(bgr, report=True)
    info["filepath"] = row["filepath"]
    info["class_name"] = row["class_name"]
    info["crop"] = row["crop"]
    rows.append(info)

calib_df = pd.DataFrame(rows)

print(f"sampled {len(calib_df)} training images\n")
print(f"deblur fired      : {calib_df['deblurred'].mean():>6.1%}")
print(f"brightness fired  : {calib_df['rebalanced'].mean():>6.1%}")
print(f"untouched         : {(~calib_df['changed']).mean():>6.1%}\n")
print(calib_df[["blur_before", "blur_after", "brightness_before", "brightness_after"]]
      .describe().loc[["mean", "50%", "min", "max"]].round(1).to_string())
print("\nfiring rate by crop:")
print(calib_df.groupby("crop")[["deblurred", "rebalanced"]].mean().round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(calib_df["blur_before"], bins=50)
axes[0].axvline(BLUR_VARIANCE_MIN, color="red", ls="--", label="BLUR_VARIANCE_MIN")
axes[0].set_title("Laplacian variance (sharpness)"); axes[0].legend()
axes[1].hist(calib_df["brightness_before"], bins=50)
axes[1].axvline(BRIGHTNESS_LOW, color="red", ls="--")
axes[1].axvline(BRIGHTNESS_HIGH, color="red", ls="--", label="accepted band")
axes[1].set_title("mean grey level (exposure)"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# eyeball the ones the gates actually touched — this is the check the numbers cannot do
touched = calib_df[calib_df["changed"]]
print(f"{len(touched)} of {len(calib_df)} sampled images were modified")

for _, row in touched.head(4).iterrows():
    bgr = cv2.imread(row["filepath"])
    cleaned, info = preprocess_image(bgr, report=True)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
    axes[0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"before — blur {info['blur_before']:.0f}, "
                      f"bright {info['brightness_before']:.0f}", fontsize=9)
    axes[1].imshow(cv2.cvtColor(cleaned, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"after — blur {info['blur_after']:.0f}, "
                      f"bright {info['brightness_after']:.0f}"
                      f"  [{'deblur ' if info['deblurred'] else ''}"
                      f"{'brightness' if info['rebalanced'] else ''}]", fontsize=9)
    for ax in axes:
        ax.axis("off")
    fig.suptitle(row["class_name"], fontsize=10)
    plt.tight_layout(); plt.show()

## 5. Lesion segmentation — rebuilt

**The bug in the old version.** One Otsu threshold on excess-green was asked to do two
different jobs: separate *leaf from background*, and separate *healthy from diseased*.
Those are not the same split. Brown necrotic tissue is not green, so it landed on the
background side of the first split and was deleted from the leaf entirely — removed from
the numerator *and* the denominator. Filling from the edge could only bring it back if it
was fully enclosed by green tissue. Margin necrosis never is, which is why your
`tomato_late_blight` sample scored 11.9% on a leaf that is close to half destroyed.

**The fix: the two jobs get two different rules.**

*Leaf vs background* is decided by **colour distance from the background**, sampled from a
strip around the image border. Distance is measured in the **a/b (chroma) channels of Lab
only** — lightness is deliberately excluded. On your images the background's chroma MAD is
about 2 while its lightness MAD is about 28, because of the cast shadow on the concrete.
Any lightness weighting pulls that shadow into the leaf and then scores it as lesion.
Brown, yellow and tan tissue all sit far from neutral grey in chroma, so they now stay
inside the leaf where they belong.

*Healthy vs diseased* is decided **inside the leaf**, by an HSV rule: green hue, adequate
saturation, positive excess-green. Everything else inside the outline is a lesion.

**Two special cases the rule needs on its own.**

Near-black necrosis has almost no chroma, so it looks like background to the first rule.
It is rescued by **enclosure**: a dark blob is kept if most of its border touches leaf
tissue, and rejected if it opens onto the background. On your sample this correctly keeps
three necrotic regions (border contact 0.73 / 0.94 / 0.87) and rejects the cast shadow
(0.34). A blind morphological close would have worked here too, but it would fill the
gaps between the lobes of a grape leaf and then count that background as lesion — the
enclosure test is shape-aware without being shape-destroying.

A **non-uniform background** (field photos, canopy, soil) breaks the border-sampling
assumption. When the sampled background's chroma MAD exceeds `BG_UNIFORM_MAD`, the code
falls back to the old excess-green mask and says so in the returned dict, so you can filter
those images rather than trusting a number produced by a broken assumption.

Severity is still computed from the **raw** image in every split. If preprocessing ran
first, a training image's severity and the same image's severity at inference would differ,
and the bands would stop being comparable.

In [ ]:
def _odd(value, minimum=3):
    value = max(minimum, int(value))
    return value + 1 if value % 2 == 0 else value


def fill_from_edge(mask):
    """Flood the background inward from all four corners; return the solid silhouette."""
    h, w = mask.shape
    flood = mask.copy()
    pad = np.zeros((h + 2, w + 2), np.uint8)
    for seed in [(0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1)]:
        if flood[seed[1], seed[0]] == 0:
            cv2.floodFill(flood, pad, seed, 255)
    return cv2.bitwise_or(mask, cv2.bitwise_not(flood))


def largest_component(mask):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n <= 1:
        return mask
    biggest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return np.where(labels == biggest, 255, 0).astype(np.uint8)


def green_mask(bgr):
    """Excess-green + Otsu. Kept only as the fallback for non-uniform backgrounds."""
    b, g, r = cv2.split(bgr.astype(np.float32))
    exg = 2.0 * g - r - b
    exg_u8 = cv2.normalize(exg, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, mask = cv2.threshold(exg_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(min(bgr.shape[:2]) * 0.012),) * 2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    return largest_component(mask)


def background_stats(lab, border=BORDER_FRACTION):
    """Median and MAD of the border strip — the model of 'what background looks like'."""
    h, w = lab.shape[:2]
    b = max(2, int(min(h, w) * border))
    frame = np.concatenate([lab[:b].reshape(-1, 3), lab[-b:].reshape(-1, 3),
                            lab[:, :b].reshape(-1, 3), lab[:, -b:].reshape(-1, 3)])
    median = np.median(frame, axis=0)
    mad = np.median(np.abs(frame - median), axis=0)
    return median, mad


def rescue_dark_tissue(lab, core, bg_median, drop=DARK_DROP,
                       enclosure=DARK_ENCLOSURE, min_area=DARK_MIN_AREA):
    """Near-black necrosis has no chroma, so the background rule cannot see it.

    Keep a dark blob if most of its border touches leaf tissue (necrosis sits IN the leaf)
    and drop it if it opens onto the background (that is a cast shadow)."""
    lightness = lab[:, :, 0]
    dark = ((lightness < bg_median[0] - drop) & (core == 0)).astype(np.uint8) * 255
    n, labels, stats, _ = cv2.connectedComponentsWithStats(dark, 8)
    ring_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    keep = np.zeros_like(dark)
    h, w = dark.shape

    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] < min_area:
            continue
        x, y = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP]
        bw, bh = stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        if x == 0 or y == 0 or x + bw >= w or y + bh >= h:
            continue                       # touches the frame -> it is the scene, not a lesion
        blob = (labels == i).astype(np.uint8) * 255
        ring = cv2.subtract(cv2.dilate(blob, ring_kernel), blob)
        ring_pixels = np.count_nonzero(ring)
        if ring_pixels and np.count_nonzero(cv2.bitwise_and(ring, core)) / ring_pixels >= enclosure:
            keep = cv2.bitwise_or(keep, blob)
    return keep


def leaf_mask(bgr):
    """The whole leaf — green, brown, yellow and black tissue alike."""
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    bg_median, bg_mad = background_stats(lab)
    chroma_mad = float(bg_mad[1] + bg_mad[2])
    uniform = chroma_mad <= BG_UNIFORM_MAD

    green = green_mask(bgr)

    if uniform:
        # chroma distance ONLY — including lightness would swallow the cast shadow
        distance = np.sqrt(((lab[:, :, 1:] - bg_median[1:]) ** 2).sum(axis=2))
        u8 = cv2.normalize(distance, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        cut, _ = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        chroma = (((u8 >= cut) & (distance >= CHROMA_FLOOR)).astype(np.uint8)) * 255
        core = largest_component(cv2.bitwise_or(chroma, green))
        core = cv2.bitwise_or(core, rescue_dark_tissue(lab, core, bg_median))
    else:
        core = largest_component(green)

    h, w = core.shape
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(min(h, w) * LEAF_CLOSE_RATIO),) * 2)
    solid = fill_from_edge(cv2.morphologyEx(fill_from_edge(core), cv2.MORPH_CLOSE, k))
    return solid, green, {"uniform_background": bool(uniform),
                          "background_chroma_mad": round(chroma_mad, 2)}


def healthy_tissue_mask(bgr, leaf):
    """Green, saturated tissue inside the leaf. Everything else inside it is a lesion."""
    hue, sat, _ = cv2.split(cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV))
    b, g, r = cv2.split(bgr.astype(np.float32))
    exg = 2.0 * g - r - b
    healthy = ((hue >= HUE_LOW) & (hue <= HUE_HIGH) & (sat >= SAT_MIN) & (exg > 0))
    return cv2.bitwise_and(healthy.astype(np.uint8) * 255, leaf)


def analyse_leaf(bgr):
    """Leaf mask, healthy mask, lesion mask, boxes (original scale) and severity %."""
    h0, w0 = bgr.shape[:2]
    scale = WORK_SIZE / max(h0, w0)
    work = (cv2.resize(bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
            if scale < 1 else bgr.copy())
    h, w = work.shape[:2]

    leaf, green, info = leaf_mask(work)
    healthy = healthy_tissue_mask(work, leaf)
    lesion = cv2.bitwise_and(leaf, cv2.bitwise_not(healthy))

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (_odd(min(h, w) * LESION_SMOOTH_RATIO),) * 2)
    lesion = cv2.morphologyEx(lesion, cv2.MORPH_OPEN, k, iterations=1)
    lesion = cv2.morphologyEx(lesion, cv2.MORPH_CLOSE, k, iterations=2)

    leaf_area = max(1, int(np.count_nonzero(leaf)))
    severity = 100.0 * np.count_nonzero(lesion) / leaf_area

    n, labels, stats, _ = cv2.connectedComponentsWithStats(lesion, 8)
    pad = max(2, int(min(h, w) * BOX_PADDING_RATIO))
    scored = []
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        ratio = area / leaf_area
        if ratio < MIN_LESION_RATIO or ratio > MAX_LESION_RATIO:
            continue
        x, y = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP]
        bw, bh = stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        if bw < 4 or bh < 4:
            continue
        scored.append((area, (max(0, x - pad), max(0, y - pad),
                              min(w - 1, x + bw + pad), min(h - 1, y + bh + pad))))

    scored.sort(key=lambda t: t[0], reverse=True)
    inv = 1.0 / scale if scale < 1 else 1.0
    boxes = [tuple(int(v * inv) for v in box) for _, box in scored[:MAX_BOXES]]

    result = {
        "leaf_mask": leaf,
        "green_mask": green,
        "healthy_mask": healthy,
        "lesion_mask": lesion,
        "boxes": boxes,
        "severity": severity,
        "work_shape": (h, w),
        # share of the frame the leaf covers. Section 11's first screen reads this to
        # decide whether there is a leaf in the picture at all.
        "leaf_fraction": float(leaf_area) / float(h * w),
    }
    result.update(info)
    return result


# every key any later section reads off analyse_leaf, checked once, here — so a missing
# one fails in this cell rather than an hour into the run
REQUIRED_ANALYSIS_KEYS = {"leaf_mask", "green_mask", "healthy_mask", "lesion_mask",
                          "boxes", "severity", "work_shape", "leaf_fraction",
                          "uniform_background", "background_chroma_mad"}
_probe_keys = set(analyse_leaf(np.full((64, 64, 3), 120, np.uint8)))
assert REQUIRED_ANALYSIS_KEYS <= _probe_keys, \
    f"analyse_leaf is missing: {sorted(REQUIRED_ANALYSIS_KEYS - _probe_keys)}"
print("analyse_leaf returns:", sorted(_probe_keys))

### 5a. Prove it on a leaf whose lesion size you already know

Real leaves have no ground truth, so a visual check can only tell you the mask looks
plausible. This cell paints a lesion of **known area** onto a synthetic leaf and checks the
measured severity against it. Five cases, and the third is the one that matters: a lesion
touching the leaf margin, which the old rule scored at **0.0%**.

If you change any threshold in section 2, re-run this cell first. It is the only place in
the notebook where severity has an answer to be wrong about.

In [ ]:
def synthetic_leaf(patches=(), size=400):
    """Green ellipse on neutral grey, with lesions of exactly known area painted on."""
    rng = np.random.default_rng(0)
    image = np.full((size, size, 3), 150, np.uint8)
    noise = rng.integers(-6, 7, image.shape).astype(np.int16)
    image = np.clip(image.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    leaf = np.zeros((size, size), np.uint8)
    cv2.ellipse(leaf, (size // 2, size // 2), (130, 165), 0, 0, 360, 255, -1)
    image[leaf > 0] = (60, 140, 70)

    truth = np.zeros((size, size), np.uint8)
    for cx, cy, rx, ry, colour in patches:
        blob = np.zeros((size, size), np.uint8)
        cv2.ellipse(blob, (cx, cy), (rx, ry), 0, 0, 360, 255, -1)
        blob = cv2.bitwise_and(blob, leaf)
        image[blob > 0] = colour
        truth = cv2.bitwise_or(truth, blob)
    return image, leaf, truth


SYNTHETIC_CASES = {
    "healthy, no lesion": [],
    "lesion in the interior": [(200, 200, 55, 55, (60, 90, 130))],
    "lesion touching the margin": [(200, 90, 70, 55, (60, 90, 130))],
    "near-black necrosis": [(200, 180, 60, 60, (28, 30, 34))],
    "two separate lesions": [(160, 160, 45, 45, (60, 90, 130)),
                             (250, 250, 40, 40, (55, 110, 140))],
}

print(f"{'case':30s} {'true':>7s} {'measured':>9s} {'error':>7s}")
print("-" * 56)
worst_error = 0.0
for name, patches in SYNTHETIC_CASES.items():
    image, leaf, truth = synthetic_leaf(patches)
    true_pct = 100.0 * np.count_nonzero(truth) / np.count_nonzero(leaf)
    measured = analyse_leaf(image)["severity"]
    error = abs(measured - true_pct)
    worst_error = max(worst_error, error)
    print(f"{name:30s} {true_pct:6.1f}% {measured:8.1f}% {error:6.1f}")

assert worst_error < 3.0, f"severity is off by {worst_error:.1f} points on synthetic ground truth"
print(f"\nworst error {worst_error:.1f} points — segmentation is measuring what it should")

### 5b. Severity level — lesion % into Healthy / Mild / Moderate / High

`severity_level(percent)` is the pure rule. `assess_severity(bgr)` is the wrapper: it runs
the mask, grades it, counts lesions, measures the biggest, and **gates on mask quality** —
a leaf mask covering almost none or almost all of the frame has failed, so the function
returns `"Unavailable"` rather than a confident wrong number.

In [ ]:
SEVERITY_ORDER = ["Unavailable", "Healthy", "Mild", "Moderate", "High"]


def severity_level(percent,
                   healthy_max=HEALTHY_SEVERITY,
                   mild_max=MILD_MAX,
                   moderate_max=MODERATE_MAX):
    """% of leaf area covered by lesions  ->  severity band."""
    if percent is None or not np.isfinite(percent):
        return "Unavailable"
    if percent < healthy_max:
        return "Healthy"
    if percent < mild_max:
        return "Mild"
    if percent < moderate_max:
        return "Moderate"
    return "High"


def leaf_mask_quality(analysis):
    """Fraction of the frame the leaf mask claims. Outside the sane range = mask failure."""
    if "leaf_fraction" in analysis:            # one source of truth, no drift
        fraction = float(analysis["leaf_fraction"])
    else:
        h, w = analysis["work_shape"]
        fraction = np.count_nonzero(analysis["leaf_mask"]) / float(max(1, h * w))
    return float(fraction), bool(MIN_LEAF_FRACTION <= fraction <= MAX_LEAF_FRACTION)


def assess_severity(bgr, analysis=None):
    """Full severity read-out for one leaf image."""
    if analysis is None:
        analysis = analyse_leaf(bgr)

    leaf_fraction, mask_ok = leaf_mask_quality(analysis)
    leaf_area = max(1, int(np.count_nonzero(analysis["leaf_mask"])))

    count, _, stats, _ = cv2.connectedComponentsWithStats(analysis["lesion_mask"], 8)
    blob_areas = [stats[i, cv2.CC_STAT_AREA] for i in range(1, count)]
    keep = [a for a in blob_areas if MIN_LESION_RATIO <= a / leaf_area <= MAX_LESION_RATIO]
    largest_pct = float(100.0 * max(keep) / leaf_area) if keep else 0.0

    percent = float(analysis["severity"]) if mask_ok else None
    level = severity_level(percent) if mask_ok else "Unavailable"

    return {
        "severity_percent": None if percent is None else round(percent, 2),
        "severity_level": level,
        "lesion_count": len(keep),
        "largest_lesion_pct": round(largest_pct, 2),
        "leaf_fraction": round(leaf_fraction, 3),
        "mask_ok": bool(mask_ok),
    }


print(manifest["class_name"].iloc[0], "->", assess_severity(_probe_bgr))

## 6. Look at it before you trust it

If the green mask is grabbing the background, or a healthy leaf shows 30% severity, fix it
here — every number downstream inherits this step. The previous version had a real problem
here: 37% of healthy images graded Mild or worse. Check the four panels per class — the
`leaf` panel should be the whole leaf including brown areas, and the `healthy tissue` panel
should be only the green parts.

In [ ]:
def show_analysis(n_per_class=1, classes=None):
    pool = manifest if classes is None else manifest[manifest["class_name"].isin(classes)]
    picks = (pool.groupby("class_name", group_keys=False)
                 .apply(lambda d: d.sample(min(n_per_class, len(d)), random_state=SEED)))

    for _, row in picks.iterrows():
        bgr = cv2.imread(row["filepath"])
        if bgr is None:
            continue
        out = analyse_leaf(bgr)

        overlay = cv2.resize(bgr, out["work_shape"][::-1], interpolation=cv2.INTER_AREA)
        red = np.zeros_like(overlay); red[:, :, 2] = 255
        overlay = np.where(out["lesion_mask"][:, :, None] > 0,
                           cv2.addWeighted(overlay, 0.45, red, 0.55, 0), overlay)

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        for ax, img, title in zip(
            axes,
            [cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), out["leaf_mask"],
             out["healthy_mask"], cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)],
            ["original", "leaf (whole leaf, incl. brown)", "healthy tissue only",
             f"lesions — {out['severity']:.1f}%  ({len(out['boxes'])} boxes)"],
        ):
            ax.imshow(img, cmap=None if img.ndim == 3 else "gray")
            ax.set_title(title, fontsize=10)
            ax.axis("off")
        flag = "" if out["uniform_background"] else "   [non-uniform background -> ExG fallback]"
        fig.suptitle(f"{row['class_name']}  "
                     f"({'healthy' if row['healthy'] else 'diseased'}){flag}")
        plt.tight_layout(); plt.show()


show_analysis(n_per_class=1)

In [ ]:
# healthy severity should sit near zero and clearly below the diseased classes
sample = manifest.groupby("healthy", group_keys=False).apply(
    lambda d: d.sample(min(120, len(d)), random_state=SEED))

rows = []
for _, row in sample.iterrows():
    bgr = cv2.imread(row["filepath"])
    if bgr is None:
        continue
    report = assess_severity(bgr)
    out = analyse_leaf(bgr)
    rows.append({"healthy": row["healthy"],
                 "class_name": row["class_name"],
                 "severity": out["severity"],
                 "severity_level": report["severity_level"],
                 "uniform_background": out["uniform_background"]})

sev = pd.DataFrame(rows)
print(sev.groupby("healthy")["severity"].describe()[["count", "mean", "50%", "75%", "max"]])

plt.figure(figsize=(7, 4))
for flag, label in [(True, "healthy"), (False, "diseased")]:
    plt.hist(sev[sev["healthy"] == flag]["severity"], bins=40, alpha=0.55, label=label)
plt.xlabel("lesion severity (% of leaf area)"); plt.ylabel("images"); plt.legend(); plt.show()

levels = pd.CategoricalDtype(SEVERITY_ORDER, ordered=True)
print(pd.crosstab(sev["healthy"], sev["severity_level"].astype(levels)).to_string())

print(f"\nborder-sampled background usable on {sev['uniform_background'].mean():.1%} of images")
separation = sev[~sev["healthy"]]["severity"].median() / max(1e-6, sev[sev["healthy"]]["severity"].median())
print(f"diseased/healthy median severity ratio: {separation:.1f}x  (higher is better)")

## 7. Dataset and loaders

`LeafDataset` takes a `preprocess` flag; `make_loader` sets it from
`split in PREPROCESS_SPLITS`, which is now all three splits. Because the flag lives on the
dataset object rather than inside the transform, there is exactly one place where a split's
preprocessing is decided, and the print-out below shows what each split got.

**Order inside `__getitem__` matters:** read → *preprocess* → augment → normalise. Cleaning
must happen before `RandomResizedCrop` and `ColorJitter`, because the gates measure blur
and exposure on the real photo. Running them after augmentation would mean the jitter
decides whether the deblur fires, which makes the decision random per epoch.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class LeafDataset(Dataset):
    def __init__(self, frame, transform, lesion_channel=False, preprocess=False):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.lesion_channel = lesion_channel
        self.preprocess = preprocess          # <- CHANGE 2 lives here

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        bgr = cv2.imread(row["filepath"])
        if bgr is None:
            bgr = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), np.uint8)

        raw = bgr
        if self.preprocess:
            bgr = preprocess_image(bgr)       # clean BEFORE augmenting

        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        tensor = self.transform(rgb)

        if self.lesion_channel:
            # mask comes off the RAW image so severity means the same thing in every split
            mask = analyse_leaf(raw)["lesion_mask"]
            mask = cv2.resize(mask, (tensor.shape[2], tensor.shape[1]),
                              interpolation=cv2.INTER_NEAREST)
            mask_t = torch.from_numpy(mask.astype(np.float32) / 255.0).unsqueeze(0)
            tensor = torch.cat([tensor, mask_t], dim=0)

        return tensor, int(row["label_id"])


def make_loader(split, training, preprocess=None):
    frame = manifest[manifest["split"] == split]
    if preprocess is None:
        preprocess = split in PREPROCESS_SPLITS          # the one decision point

    dataset = LeafDataset(frame, train_tf if training else eval_tf,
                          USE_LESION_CHANNEL, preprocess)

    sampler, shuffle = None, training
    if training and BALANCE_CLASSES:
        counts = frame["label_id"].value_counts()
        weights = frame["label_id"].map(lambda i: 1.0 / counts[i]).values
        sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights), replacement=True)
        shuffle = False

    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, sampler=sampler,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=training)


train_loader = make_loader("train", True)
val_loader   = make_loader("val", False)
test_loader  = make_loader("test", False)

print("batches — train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))
print("\npreprocessing applied to:")
for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    print(f"  {name:6s} {loader.dataset.preprocess}")
for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    assert loader.dataset.preprocess == (name in PREPROCESS_SPLITS), \
        f"{name} loader disagrees with PREPROCESS_SPLITS"
print("\nloaders match PREPROCESS_SPLITS")

## 8. The CNN

EfficientNet-B0, ImageNet transfer learning. With a 4th channel the first conv is widened
and the extra channel initialised at zero, so the pretrained RGB weights start untouched.

In [ ]:
def build_model(num_classes, backbone=BACKBONE, in_channels=3):
    """EfficientNet-B0 transfer learning — the only backbone this notebook builds."""
    if backbone != "efficientnet_b0":
        raise ValueError(
            f"Unsupported backbone: {backbone!r}. This notebook only builds efficientnet_b0."
        )

    model = torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1")
    first = model.features[0][0]
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    if in_channels != 3:
        wide = nn.Conv2d(in_channels, first.out_channels, first.kernel_size,
                         first.stride, first.padding, bias=first.bias is not None)
        with torch.no_grad():
            wide.weight[:, :3] = first.weight
            wide.weight[:, 3:] = 0.0
        model.features[0][0] = wide

    return model


model = build_model(NUM_CLASSES, BACKBONE, 4 if USE_LESION_CHANNEL else 3).to(DEVICE)
print(BACKBONE, "|", round(sum(p.numel() for p in model.parameters()) / 1e6, 4), "M params",
      "|", NUM_CLASSES, "classes")

## 9. Train

The checkpoint now carries a `config` block — image size, normalisation, confidence gate,
the crops kept and the preprocessing settings. The Flask app reads these instead of
hard-coding them, so a retrain with different settings cannot silently desynchronise from
deployment.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE * 5,
    steps_per_epoch=max(1, len(train_loader)), epochs=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

RUN_CONFIG = {
    "backbone": BACKBONE,
    "img_size": IMAGE_SIZE,
    "mean": MEAN,
    "std": STD,
    "confidence_gate": CONFIDENCE_GATE,
    "in_channels": 4 if USE_LESION_CHANNEL else 3,
    "crops": sorted(CROPS),
    "num_classes": NUM_CLASSES,
    "preprocess_splits": sorted(PREPROCESS_SPLITS),
    "preprocess": {
        "blur_variance_min": BLUR_VARIANCE_MIN,
        "unsharp_radius": UNSHARP_RADIUS,
        "unsharp_amount": UNSHARP_AMOUNT,
        "brightness_target": BRIGHTNESS_TARGET,
        "brightness_low": BRIGHTNESS_LOW,
        "brightness_high": BRIGHTNESS_HIGH,
        "gamma_limits": list(GAMMA_LIMITS),
        "clahe_clip": CLAHE_CLIP,
        "clahe_grid": CLAHE_GRID,
    },
    "severity_bands": {
        "healthy_max": HEALTHY_SEVERITY, "mild_max": MILD_MAX, "moderate_max": MODERATE_MAX,
    },
}


def run_epoch(loader, training):
    model.train() if training else model.eval()
    total_loss = correct = seen = 0

    with torch.set_grad_enabled(training):
        for images, targets in loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, targets)

            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            total_loss += loss.item() * targets.size(0)
            correct += (logits.argmax(1) == targets).sum().item()
            seen += targets.size(0)

    return total_loss / max(1, seen), correct / max(1, seen)


if TRAIN_MODEL:
    history, best_accuracy = [], 0.0

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_accuracy = run_epoch(train_loader, True)
        val_loss, val_accuracy = run_epoch(val_loader, False)
        history.append((epoch, train_loss, train_accuracy, val_loss, val_accuracy))

        flag = ""
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            torch.save({"state_dict": model.state_dict(),
                        "class_to_id": class_to_id,
                        "backbone": BACKBONE,
                        "in_channels": 4 if USE_LESION_CHANNEL else 3,
                        "config": RUN_CONFIG,
                        "epoch": epoch,
                        "val_accuracy": val_accuracy}, CHECKPOINT_PATH)
            flag = "  <- saved"

        print(f"epoch {epoch:02d} | train {train_loss:.3f}/{train_accuracy:.3f}"
              f" | val {val_loss:.3f}/{val_accuracy:.3f}{flag}")

    hist = pd.DataFrame(history, columns=["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    hist.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], title="loss")
    hist.plot(x="epoch", y=["train_acc", "val_acc"], ax=axes[1], title="accuracy")
    plt.show()
    print("Best validation accuracy:", round(best_accuracy, 4))

## 10. Test-set evaluation

This is the headline number. The test set is **not** preprocessed, so this measures the
model in the condition it will actually meet: trained on cleaned images, deployed on
whatever the farmer's phone produced.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()


@torch.no_grad()
def evaluate(loader):
    true_labels, predicted_labels = [], []
    for images, targets in loader:
        logits = model(images.to(DEVICE))
        predicted_labels.extend(logits.argmax(1).cpu().tolist())
        true_labels.extend(targets.tolist())
    return true_labels, predicted_labels


true_labels, predicted_labels = evaluate(test_loader)

present = sorted(set(true_labels) | set(predicted_labels))
print(classification_report(
    true_labels, predicted_labels, labels=present,
    target_names=[id_to_class[i] for i in present], digits=3, zero_division=0))

raw_accuracy = float(np.mean(np.array(true_labels) == np.array(predicted_labels)))
raw_macro_f1 = f1_score(true_labels, predicted_labels, average="macro", zero_division=0)

matrix = confusion_matrix(true_labels, predicted_labels, labels=present)
plt.figure(figsize=(min(18, 0.5 * len(present) + 4),) * 2)
plt.imshow(matrix, cmap="viridis")
plt.xticks(range(len(present)), [id_to_class[i] for i in present], rotation=90, fontsize=8)
plt.yticks(range(len(present)), [id_to_class[i] for i in present], fontsize=8)
plt.xlabel("predicted"); plt.ylabel("true"); plt.colorbar(); plt.tight_layout(); plt.show()

per_class = pd.DataFrame({
    "class": [id_to_class[i] for i in present],
    "f1": f1_score(true_labels, predicted_labels, labels=present,
                   average=None, zero_division=0).round(3),
    "support": [int((np.array(true_labels) == i).sum()) for i in present],
}).sort_values("f1")
display(per_class)

print(f"test accuracy {raw_accuracy:.4f} | macro-F1 {raw_macro_f1:.4f}")
print(f"weakest class: {per_class.iloc[0]['class']} at F1 {per_class.iloc[0]['f1']:.3f}")

### 10b. Proof that preprocessing did not leak

Two properties make a transform safe to apply to every split, and this cell checks both
rather than asserting them in prose.

**Per-image.** Running an image through `preprocess_image` on its own must give exactly the
same result as running it inside a batch of other images. If it did not, some cross-image
statistic would be involved — and that is the shape leakage takes.

**Deterministic.** The same input must give the same output every time. A transform with
randomness in it cannot be reproduced at inference.

The cell also scores the test set with preprocessing switched **off**, purely as
information: it tells you how much of your accuracy the clean-up is responsible for.
Because preprocessing now runs everywhere, the ON row is the honest headline number and
matches what the deployed app will see, as long as the app runs the same transform.

In [ ]:
# --- property 1: per-image, no cross-image statistics ---
probe = manifest.sample(min(12, len(manifest)), random_state=SEED)
images = [cv2.imread(p) for p in probe["filepath"]]
images = [im for im in images if im is not None]

alone = [preprocess_image(im) for im in images]
shuffled = list(reversed(images))
together = list(reversed([preprocess_image(im) for im in shuffled]))

per_image = all(np.array_equal(a, b) for a, b in zip(alone, together))
print("per-image (order-independent):", per_image)

# --- property 2: deterministic ---
deterministic = all(np.array_equal(preprocess_image(im), preprocess_image(im)) for im in images)
print("deterministic (same in, same out):", deterministic)

assert per_image, "preprocessing depends on the other images in the batch — that IS leakage"
assert deterministic, "preprocessing is not deterministic"
print("\npreprocessing is a fixed per-image transform — safe on val and test")

# --- how much is the clean-up worth? ---
raw_loader = make_loader("test", False, preprocess=False)
true_raw, pred_raw = evaluate(raw_loader)
off_accuracy = float(np.mean(np.array(true_raw) == np.array(pred_raw)))
off_macro_f1 = f1_score(true_raw, pred_raw, average="macro", zero_division=0)

print("\n" + pd.DataFrame({
    "preprocessing": ["ON  (headline — matches deployment)", "OFF (diagnostic only)"],
    "accuracy": [round(raw_accuracy, 4), round(off_accuracy, 4)],
    "macro_f1": [round(raw_macro_f1, 4), round(off_macro_f1, 4)],
}).to_string(index=False))
print(f"\npreprocessing is worth {raw_accuracy - off_accuracy:+.4f} accuracy on the test set")
print("If the deployed app does NOT run preprocess_image, the OFF row is what you will get.")

## 11. Refusing what the model cannot classify

You asked for two behaviours: refuse leaf types that are not in the training set, and
return a plain white card when confidence is under 80%. The second is easy. The first is
the hard problem in this notebook, and it is worth being precise about why.

**Softmax cannot detect an unseen class.** The final layer produces a distribution over
*your 13 classes only*. Feed it a wheat leaf and it does not output "none of these" — it
outputs the closest of the 13, often at 0.95+. Confidence thresholding catches blurry and
ambiguous images; it does not catch confidently-wrong ones. Shipping only the 0.80 gate
would give you a system that refuses good photos and accepts foreign ones.

**So three independent screens run, in this order:**

1. **Is there a leaf?** Reuses the section 5 mask. A photo of a hand, a wall or a whole
   plant fails the `leaf_fraction` range and is rejected before the CNN runs at all. This
   is the cheapest screen and it catches the most obviously wrong inputs.

2. **Does this look like the training data?** The 1280-dimensional penultimate features of
   EfficientNet-B0 are summarised as a class-conditional Gaussian fitted on the **training
   split only**. An image's score is its Mahalanobis distance to the nearest class centre.
   Grape and tomato leaves land close to a centre; a wheat leaf, a flower, or a photo of
   a screen lands far away — even when softmax is confident. This is the screen that
   actually does the job you asked for.

3. **Is the classifier confident?** The 0.80 gate.

**The threshold is calibrated, not guessed.** Section 11b sets the distance cut-off from
the **validation** split so that `OOD_RETENTION` (99.5%) of genuine grape/tomato images
pass. That makes the false-rejection rate a number you choose rather than a surprise. It
also means the cut-off is fitted on data the model did not train on and that test never
sees — no leakage.

**What this cannot do.** Mahalanobis screening is a good detector, not a guarantee. A
diseased leaf of a *closely related* species may sit inside the training distribution and
pass. Section 11c measures the real rate against whatever negatives you can supply — point
`EXTRA_OOD_DIR` at your old wheat folder if you still have it, and read the number instead
of trusting the method.

In [ ]:
# ---------- 11a. penultimate features ----------

@torch.no_grad()
def extract_features(loader, limit=None):
    """1280-d penultimate activations of EfficientNet-B0, plus logits and labels."""
    model.eval()
    features, logits_all, labels_all, seen = [], [], [], 0
    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        maps = model.features(images)
        pooled = torch.flatten(model.avgpool(maps), 1)
        logits = model.classifier(pooled)
        features.append(pooled.cpu().numpy())
        logits_all.append(logits.cpu().numpy())
        labels_all.append(targets.numpy())
        seen += len(targets)
        if limit and seen >= limit:
            break
    return (np.concatenate(features), np.concatenate(logits_all), np.concatenate(labels_all))


print("extracting training features...")
train_eval_loader = DataLoader(
    LeafDataset(manifest[manifest["split"] == "train"], eval_tf,
                USE_LESION_CHANNEL, "train" in PREPROCESS_SPLITS),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

train_features, _, train_labels_arr = extract_features(train_eval_loader, limit=8000)
print("features:", train_features.shape)


def fit_gaussian(features, labels, num_classes, shrinkage=FEATURE_SHRINKAGE):
    """Class means + one shared covariance (the standard Mahalanobis OOD setup)."""
    means = np.stack([features[labels == c].mean(axis=0) if (labels == c).any()
                      else features.mean(axis=0) for c in range(num_classes)])
    centred = features - means[labels]
    covariance = np.cov(centred, rowvar=False)
    covariance += shrinkage * np.trace(covariance) / covariance.shape[0] * np.eye(covariance.shape[0])
    return means, np.linalg.pinv(covariance)


CLASS_MEANS, PRECISION = fit_gaussian(train_features, train_labels_arr, NUM_CLASSES)
print("fitted", CLASS_MEANS.shape[0], "class centres in", CLASS_MEANS.shape[1], "dimensions")


def mahalanobis_score(features, means=None, precision=None):
    """Distance to the NEAREST class centre. Small = looks like training data."""
    means = CLASS_MEANS if means is None else means
    precision = PRECISION if precision is None else precision
    best = None
    for centre in means:
        delta = features - centre
        distance = np.einsum("ij,jk,ik->i", delta, precision, delta)
        best = distance if best is None else np.minimum(best, distance)
    return np.sqrt(np.maximum(best, 0.0))

In [ ]:
# ---------- 11b. calibrate the cut-off on VALIDATION, never on test ----------

val_features, val_logits, _ = extract_features(val_loader)
val_scores = mahalanobis_score(val_features)

OOD_THRESHOLD = float(np.quantile(val_scores, OOD_RETENTION))

print(f"validation Mahalanobis distance:")
for q in [0.50, 0.90, 0.99, OOD_RETENTION, 1.0]:
    print(f"   p{q * 100:5.1f} : {np.quantile(val_scores, q):8.2f}")
print(f"\nOOD_THRESHOLD = {OOD_THRESHOLD:.2f}  "
      f"(rejects {1 - OOD_RETENTION:.1%} of genuine validation images)")

plt.figure(figsize=(7, 4))
plt.hist(val_scores, bins=60, alpha=0.7, label="validation (in-distribution)")
plt.axvline(OOD_THRESHOLD, color="red", ls="--", label=f"threshold {OOD_THRESHOLD:.1f}")
plt.xlabel("Mahalanobis distance to nearest class centre"); plt.ylabel("images")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ---------- 11c. the screens, and the white card ----------

REJECT_MESSAGES = {
    "no_leaf": "Cannot classify - no leaf detected in the image",
    "unknown_leaf": "Cannot classify - this leaf type is not in the training dataset",
    "low_confidence": "Cannot classify - image is unclear or leaf type not recognized",
}


def render_reject_card(message, detail="", size=REJECT_CARD_SIZE):
    """Plain white image carrying the refusal text. This is the returned output."""
    card = np.full((size, size, 3), 255, np.uint8)
    font, scale, thickness = cv2.FONT_HERSHEY_SIMPLEX, 0.62, 2

    words, lines, current = message.split(), [], ""
    for word in words:
        trial = f"{current} {word}".strip()
        if cv2.getTextSize(trial, font, scale, thickness)[0][0] > size - 60 and current:
            lines.append(current); current = word
        else:
            current = trial
    lines.append(current)

    y = size // 2 - (len(lines) - 1) * 18 - (20 if detail else 0)
    for line in lines:
        (tw, th), _ = cv2.getTextSize(line, font, scale, thickness)
        cv2.putText(card, line, ((size - tw) // 2, y), font, scale, (40, 40, 40),
                    thickness, cv2.LINE_AA)
        y += th + 16

    if detail:
        (tw, th), _ = cv2.getTextSize(detail, font, 0.45, 1)
        cv2.putText(card, detail, ((size - tw) // 2, y + 14), font, 0.45,
                    (130, 130, 130), 1, cv2.LINE_AA)
    return card


@torch.no_grad()
def screen_and_classify(bgr, analysis=None):
    """Run the three screens. Returns a verdict dict; 'accepted' False means refuse."""
    analysis = analyse_leaf(bgr) if analysis is None else analysis

    verdict = {"accepted": False, "reason": None, "class_id": None, "disease_name": None,
               "confidence": None, "ood_distance": None,
               "leaf_fraction": round(float(analysis["leaf_fraction"]), 4)}

    # screen 1 — is there a leaf at all
    if not (VEGETATION_MIN <= analysis["leaf_fraction"] <= VEGETATION_MAX):
        verdict["reason"] = "no_leaf"
        return verdict

    prepared = preprocess_image(bgr) if PREPROCESS_SPLITS else bgr
    tensor = eval_tf(cv2.cvtColor(prepared, cv2.COLOR_BGR2RGB))
    if USE_LESION_CHANNEL:
        mask = cv2.resize(analysis["lesion_mask"], (tensor.shape[2], tensor.shape[1]),
                          interpolation=cv2.INTER_NEAREST)
        tensor = torch.cat([tensor,
                            torch.from_numpy(mask.astype(np.float32) / 255.0).unsqueeze(0)], 0)

    batch = tensor.unsqueeze(0).to(DEVICE)
    pooled = torch.flatten(model.avgpool(model.features(batch)), 1)
    logits = model.classifier(pooled)
    probabilities = torch.softmax(logits, 1)[0].cpu().numpy()

    distance = float(mahalanobis_score(pooled.cpu().numpy())[0])
    class_id = int(probabilities.argmax())
    confidence = float(probabilities[class_id])

    verdict.update({"class_id": class_id, "disease_name": id_to_class[class_id],
                    "confidence": round(confidence, 4),
                    "ood_distance": round(distance, 2)})

    # screen 2 — does it look like the training distribution
    if distance > OOD_THRESHOLD:
        verdict["reason"] = "unknown_leaf"
        return verdict

    # screen 3 — is the classifier confident
    if confidence < CONFIDENCE_GATE:
        verdict["reason"] = "low_confidence"
        return verdict

    verdict["accepted"] = True
    return verdict


# sanity: the card renders, and obvious non-leaves are refused
_card = render_reject_card(REJECT_MESSAGES["low_confidence"], "confidence 0.41 / threshold 0.80")
print("reject card:", _card.shape, "| all-white background:",
      bool((_card[:12, :12] == 255).all()))
plt.figure(figsize=(4, 4)); plt.imshow(cv2.cvtColor(_card, cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.show()

In [ ]:
# ---------- 11d. how often does each screen fire, and on what ----------

# cost side: genuine grape/tomato images that get refused
id_sample = manifest[manifest["split"] == "test"].sample(
    min(200, (manifest["split"] == "test").sum()), random_state=SEED)

id_rows = []
for _, row in id_sample.iterrows():
    bgr = cv2.imread(row["filepath"])
    if bgr is None:
        continue
    v = screen_and_classify(bgr)
    v["true_class"] = row["class_name"]
    id_rows.append(v)

id_frame = pd.DataFrame(id_rows)
print(f"IN-DISTRIBUTION ({len(id_frame)} real grape/tomato test images)")
print(f"   accepted        : {id_frame['accepted'].mean():.1%}")
print(f"   refused         : {(~id_frame['accepted']).mean():.1%}")
print(id_frame[~id_frame["accepted"]]["reason"].value_counts().to_string() or "   none")
accepted = id_frame[id_frame["accepted"]]
if len(accepted):
    print(f"   accuracy on accepted images: "
          f"{(accepted['disease_name'] == accepted['true_class']).mean():.3f}")

# benefit side: things that should be refused
def synthetic_negatives(n=40):
    """Not leaves: noise, flat colour, text-like stripes, sky gradients."""
    rng = np.random.default_rng(SEED)
    out = []
    for i in range(n):
        kind = i % 4
        if kind == 0:
            img = rng.integers(0, 256, (300, 300, 3), dtype=np.uint8)
        elif kind == 1:
            img = np.full((300, 300, 3), rng.integers(0, 256, 3).tolist(), np.uint8)
        elif kind == 2:
            img = np.full((300, 300, 3), 240, np.uint8)
            for y in range(20, 280, 22):
                cv2.line(img, (30, y), (270, y), (40, 40, 40), 3)
        else:
            img = np.zeros((300, 300, 3), np.uint8)
            for y in range(300):
                img[y, :] = (200 - y // 3, 150 - y // 6, 90)
        out.append(img)
    return out

ood_images = synthetic_negatives()
ood_source = "synthetic non-leaf images"

if EXTRA_OOD_DIR:
    extra = [cv2.imread(str(p)) for p in sorted(Path(EXTRA_OOD_DIR).rglob("*"))
             if p.suffix.lower() in IMAGE_EXTENSIONS][:300]
    extra = [im for im in extra if im is not None]
    if extra:
        ood_images = extra
        ood_source = f"{EXTRA_OOD_DIR} ({len(extra)} images)"

ood_rows = [screen_and_classify(im) for im in ood_images]
ood_frame = pd.DataFrame(ood_rows)
print(f"\nOUT-OF-DISTRIBUTION ({ood_source})")
print(f"   refused         : {(~ood_frame['accepted']).mean():.1%}   <- want this high")
print(ood_frame[~ood_frame["accepted"]]["reason"].value_counts().to_string() or "   none")
if ood_frame["accepted"].any():
    leaked = ood_frame[ood_frame["accepted"]]
    print(f"\n   {len(leaked)} slipped through, e.g. "
          f"{leaked.iloc[0]['disease_name']} at {leaked.iloc[0]['confidence']:.2f} "
          f"(distance {leaked.iloc[0]['ood_distance']})")
    print("   If this number is high with EXTRA_OOD_DIR set, lower OOD_RETENTION.")

print("\nNOTE: synthetic negatives are easy. The honest test is EXTRA_OOD_DIR pointed at")
print("real leaves of a species you did not train on — wheat, if you still have it.")

## 12. Inference — screen first, then classify

Every image goes through the section 11 screens first. An image that fails any of them
returns **one output: the white card** — no class name, no severity, no annotated leaf.
A severity percentage attached to a refused diagnosis is exactly the kind of half-answer
someone acts on.

An image that passes gets the full result: annotated leaf, class, confidence, severity.

In [ ]:
ANNOTATED_DIR = PREDICTION_ROOT / "annotated_images"
MASK_DIR      = PREDICTION_ROOT / "masks"
REJECTED_DIR  = PREDICTION_ROOT / "rejected"
for folder in [ANNOTATED_DIR, MASK_DIR, REJECTED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def readable_name(name):
    return str(name).replace("_", " ").replace("-", " ").strip().title()


def predict_image(path, save_masks=False):
    bgr = cv2.imread(str(path))
    if bgr is None:
        return None

    name = Path(path).name
    analysis = analyse_leaf(bgr)                 # raw image: severity means the same thing
    verdict = screen_and_classify(bgr, analysis)

    base = {
        "image": name,
        "accepted": verdict["accepted"],
        "reason": verdict["reason"],
        "confidence": verdict["confidence"],
        "ood_distance": verdict["ood_distance"],
        "leaf_fraction": verdict["leaf_fraction"],
    }

    # ---- refused: ONE white image, nothing else ----
    if not verdict["accepted"]:
        detail = {
            "no_leaf": f"leaf covers {verdict['leaf_fraction']:.1%} of the frame",
            "unknown_leaf": f"distance {verdict['ood_distance']} / threshold {OOD_THRESHOLD:.1f}",
            "low_confidence": (f"confidence {verdict['confidence']:.2f} / "
                               f"threshold {CONFIDENCE_GATE:.2f}"
                               if verdict["confidence"] is not None else ""),
        }[verdict["reason"]]
        card = render_reject_card(REJECT_MESSAGES[verdict["reason"]], detail)
        cv2.imwrite(str(REJECTED_DIR / name), card)
        base.update({"disease_name": None, "severity_percent": None,
                     "severity_level": None, "message": REJECT_MESSAGES[verdict["reason"]]})
        return base

    # ---- accepted: the full result ----
    severity = assess_severity(bgr, analysis)
    disease = verdict["disease_name"]

    annotated = bgr.copy()
    if analysis["severity"] >= HEALTHY_SEVERITY and analysis["boxes"]:
        lesion_full = cv2.resize(analysis["lesion_mask"], (bgr.shape[1], bgr.shape[0]),
                                 interpolation=cv2.INTER_NEAREST)
        red_layer = np.zeros_like(bgr)
        red_layer[:, :, 2] = 255
        annotated = np.where(lesion_full[:, :, None] > 0,
                             cv2.addWeighted(annotated, 0.4, red_layer, 0.6, 0),
                             annotated)

    label = (f"{readable_name(disease)}  {verdict['confidence']:.2f}  |  "
             f"{severity['severity_level']}")
    if severity["severity_percent"] is not None:
        label += f" ({severity['severity_percent']:.1f}%)"
    (tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
    cv2.rectangle(annotated, (0, 0), (tw + 16, th + baseline + 12), (0, 0, 0), -1)
    cv2.putText(annotated, label, (8, th + 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)

    cv2.imwrite(str(ANNOTATED_DIR / name), annotated)
    if save_masks:
        stem = Path(path).stem
        cv2.imwrite(str(MASK_DIR / f"{stem}_leaf.png"), analysis["leaf_mask"])
        cv2.imwrite(str(MASK_DIR / f"{stem}_lesion.png"), analysis["lesion_mask"])

    base.update({
        "disease_name": disease,
        "class_id": verdict["class_id"],
        "severity_percent": severity["severity_percent"],
        "severity_level": severity["severity_level"],
        "lesion_count": severity["lesion_count"],
        "largest_lesion_pct": severity["largest_lesion_pct"],
        "mask_ok": severity["mask_ok"],
        "uniform_background": analysis["uniform_background"],
        "n_boxes": len(analysis["boxes"]),
        "message": None,
    })
    return base

In [ ]:
# one image, in full — read this before trusting the batch
_demo_row = manifest[(manifest["split"] == "test") & (~manifest["healthy"])].sample(
    1, random_state=SEED).iloc[0]
_demo = predict_image(_demo_row["filepath"], save_masks=True)

print("true class :", _demo_row["class_name"])
print("accepted   :", _demo["accepted"], "" if _demo["accepted"] else f"({_demo['reason']})")
print("predicted  :", _demo["disease_name"], f"conf {_demo['confidence']}")
print("ood dist   :", _demo["ood_distance"], "/ threshold", round(OOD_THRESHOLD, 2))
if _demo["accepted"]:
    print("severity   :", _demo["severity_level"], _demo["severity_percent"], "%")

_folder = ANNOTATED_DIR if _demo["accepted"] else REJECTED_DIR
plt.figure(figsize=(6, 5))
plt.imshow(cv2.cvtColor(cv2.imread(str(_folder / Path(_demo_row["filepath"]).name)),
                        cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.show()

# and a refusal, end to end, on a deliberately unusable input
_junk_path = PREDICTION_ROOT / "_junk_probe.png"
cv2.imwrite(str(_junk_path),
            np.random.default_rng(0).integers(0, 256, (300, 300, 3), dtype=np.uint8))
_junk = predict_image(_junk_path)
print("\njunk image ->", _junk["reason"], "|", _junk["message"])
plt.figure(figsize=(5, 5))
plt.imshow(cv2.cvtColor(cv2.imread(str(REJECTED_DIR / _junk_path.name)), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.show()

In [ ]:
# batch over the test split
test_frame = manifest[manifest["split"] == "test"]
subset = test_frame.sample(min(300, len(test_frame)), random_state=SEED)

prediction_rows = []
for count, (_, row) in enumerate(subset.iterrows(), start=1):
    result = predict_image(row["filepath"])
    if result:
        result["true_class"] = row["class_name"]
        prediction_rows.append(result)
    if count % 50 == 0:
        print("processed", count)

predictions = pd.DataFrame(prediction_rows)
predictions["correct"] = ((predictions["disease_name"] == predictions["true_class"])
                          & predictions["accepted"])
predictions.to_csv(PREDICTION_ROOT / "predictions.csv", index=False)

ok = predictions[predictions["accepted"]]
print(f"\nimages   : {len(predictions)}")
print(f"accepted : {predictions['accepted'].mean():.1%}")
print(f"refused  : {(~predictions['accepted']).mean():.1%}")
if (~predictions["accepted"]).any():
    print(predictions[~predictions["accepted"]]["reason"].value_counts().to_string())
print(f"\naccuracy on accepted images: {ok['correct'].mean():.3f}  ({len(ok)} images)")
print(f"mean confidence            : {ok['confidence'].mean():.3f}")
print(f"leaf mask usable           : {ok['mask_ok'].mean():.1%}")
print("\nseverity bands (accepted only):")
print(ok["severity_level"].value_counts().to_string())
print(f"\nwhite cards written to {REJECTED_DIR}")

## 13. Package everything and zip it

`preprocessing.py` and `severity.py` are generated with `inspect.getsource`, straight
from the functions defined above. That is deliberate: hand-copying the code into a module
is how the notebook and the deployed app drift apart, and a drifted preprocessing module
means the app cleans images differently from the way the model was trained. Generating the
module from the live objects makes drift impossible, and the import test below proves the
module reproduces the notebook byte-for-byte on a real image.

In [ ]:
def build_module(path, header, functions, constants):
    """Write a .py module whose function bodies come from this notebook's live objects."""
    parts = [header, ""]
    parts += [f"{name} = {value!r}" for name, value in constants.items()]
    parts += ["", ""]
    for f in functions:
        try:
            parts.append(inspect.getsource(f))
        except OSError as exc:
            raise RuntimeError(
                f"Could not read the source of {f.__name__}(). This happens when the cell "
                f"that defines it was not run in this kernel session (a restart, or the "
                f"function came from an import). Re-run the section that defines "
                f"{f.__name__}, then re-run this cell."
            ) from exc
    Path(path).write_text("\n".join(parts), encoding="utf-8")
    return Path(path)


preprocessing_py = build_module(
    EXPORT_ROOT / "preprocessing.py",
    header=('"""Blur removal and brightness correction — generated from notebook 01 v3.\n\n'
            'Applied to the TRAINING split only during training. If you call this at\n'
            'inference time, say so in your results: it changes the input distribution.\n'
            '"""\n'
            "import math\n\nimport cv2\nimport numpy as np"),
    functions=[blur_score, brightness_score, remove_blur, correct_brightness, preprocess_image],
    constants={
        "BLUR_VARIANCE_MIN": BLUR_VARIANCE_MIN,
        "UNSHARP_RADIUS": UNSHARP_RADIUS,
        "UNSHARP_AMOUNT": UNSHARP_AMOUNT,
        "BRIGHTNESS_TARGET": BRIGHTNESS_TARGET,
        "BRIGHTNESS_LOW": BRIGHTNESS_LOW,
        "BRIGHTNESS_HIGH": BRIGHTNESS_HIGH,
        "GAMMA_LIMITS": GAMMA_LIMITS,
        "CLAHE_CLIP": CLAHE_CLIP,
        "CLAHE_GRID": CLAHE_GRID,
    })

severity_py = build_module(
    EXPORT_ROOT / "severity.py",
    header=('"""Green-channel lesion mask and severity grading — generated from notebook 01 v3."""\n'
            "import cv2\nimport numpy as np"),
    functions=[_odd, fill_from_edge, largest_component, green_mask, background_stats,
               rescue_dark_tissue, leaf_mask, healthy_tissue_mask, analyse_leaf,
               severity_level, leaf_mask_quality, assess_severity],
    constants={
        "WORK_SIZE": WORK_SIZE, "BORDER_FRACTION": BORDER_FRACTION,
        "CHROMA_FLOOR": CHROMA_FLOOR, "BG_UNIFORM_MAD": BG_UNIFORM_MAD,
        "DARK_DROP": DARK_DROP, "DARK_ENCLOSURE": DARK_ENCLOSURE,
        "DARK_MIN_AREA": DARK_MIN_AREA, "LEAF_CLOSE_RATIO": LEAF_CLOSE_RATIO,
        "LESION_SMOOTH_RATIO": LESION_SMOOTH_RATIO,
        "HUE_LOW": HUE_LOW, "HUE_HIGH": HUE_HIGH, "SAT_MIN": SAT_MIN,
        "MIN_LESION_RATIO": MIN_LESION_RATIO, "MAX_LESION_RATIO": MAX_LESION_RATIO,
        "MAX_BOXES": MAX_BOXES, "BOX_PADDING_RATIO": BOX_PADDING_RATIO,
        "HEALTHY_SEVERITY": HEALTHY_SEVERITY,
        "MILD_MAX": MILD_MAX, "MODERATE_MAX": MODERATE_MAX,
        "MIN_LEAF_FRACTION": MIN_LEAF_FRACTION, "MAX_LEAF_FRACTION": MAX_LEAF_FRACTION,
        "SEVERITY_ORDER": SEVERITY_ORDER,
    })

RUN_CONFIG["rejection"] = {
    "confidence_gate": CONFIDENCE_GATE,
    "ood_threshold": round(float(OOD_THRESHOLD), 4),
    "ood_retention": OOD_RETENTION,
    "vegetation_min": VEGETATION_MIN,
    "vegetation_max": VEGETATION_MAX,
    "messages": REJECT_MESSAGES,
}
np.savez(EXPORT_ROOT / "ood_gaussian.npz",
         class_means=CLASS_MEANS, precision=PRECISION,
         threshold=np.array([OOD_THRESHOLD]))

(EXPORT_ROOT / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")

# the numbers, so the zip is self-describing without re-running anything
metrics = {
    "num_classes": NUM_CLASSES,
    "classes": class_names,
    "images": {"total": int(len(manifest)),
               **{k: int(v) for k, v in manifest["split"].value_counts().items()}},
    "preprocess_splits": sorted(PREPROCESS_SPLITS),
    "best_val_accuracy": round(float(best_accuracy), 4) if TRAIN_MODEL else None,
    "test_accuracy": round(float(raw_accuracy), 4),
    "test_macro_f1": round(float(raw_macro_f1), 4),
    "test_accuracy_without_preprocessing": round(float(off_accuracy), 4),
}
(EXPORT_ROOT / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

if TRAIN_MODEL:
    hist.to_csv(EXPORT_ROOT / "training_history.csv", index=False)
per_class.to_csv(EXPORT_ROOT / "per_class_f1.csv", index=False)
predictions.to_csv(EXPORT_ROOT / "predictions.csv", index=False)

for path in sorted(EXPORT_ROOT.iterdir()):
    print(f"{path.name:26s} {path.stat().st_size:>9,} bytes")

In [ ]:
# equivalence test: the exported module must reproduce the notebook exactly
import sys
sys.path.insert(0, str(EXPORT_ROOT))

import preprocessing as _pp
import importlib
importlib.reload(_pp)

check = manifest.sample(min(25, len(manifest)), random_state=SEED)
mismatches = 0
for _, row in check.iterrows():
    bgr = cv2.imread(row["filepath"])
    if bgr is None:
        continue
    if not np.array_equal(preprocess_image(bgr), _pp.preprocess_image(bgr)):
        mismatches += 1

assert mismatches == 0, f"{mismatches} images preprocess differently in the exported module"
print(f"preprocessing.py matches the notebook on {len(check)} images")

# and the checkpoint carries everything the app needs
saved = torch.load(CHECKPOINT_PATH, map_location="cpu")
print("\ncheckpoint keys :", sorted(saved.keys()))
print("config keys     :", sorted(saved["config"].keys()))
assert saved["config"]["num_classes"] == NUM_CLASSES
assert set(saved["class_to_id"]) == set(class_names)
print("checkpoint consistent with the class index")

In [ ]:
zip_base = "/kaggle/working/PlantCare_CNN_v6"      # no trailing slash: make_archive
zip_path = shutil.make_archive(zip_base, "zip",     # would otherwise emit ".zip"
                               root_dir=str(PROJECT_ROOT))

import zipfile
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()

print(f"{zip_path}  ({Path(zip_path).stat().st_size / 1e6:.1f} MB, {len(names)} files)\n")
buckets = defaultdict(int)
for name in names:
    buckets[name.split("/")[0] if "/" in name else "(root)"] += 1
for folder, count in sorted(buckets.items()):
    print(f"  {folder:24s} {count:5d} files")

expected = ["best_cnn.pt", "manifest.csv", "class_index.json",
            "artifacts/preprocessing.py", "artifacts/severity.py",
            "artifacts/run_config.json", "artifacts/metrics.json",
            "artifacts/ood_gaussian.npz"]
missing = [e for e in expected if e not in names]
assert not missing, f"zip is missing: {missing}"
print("\nall expected artifacts present in the zip")

---

### Where this will break, and what to do about it

1. **The leakage audit is the number to read first.** If section 3c reports cross-split
   duplicate groups, every accuracy figure you have quoted from earlier runs was
   optimistic. This run repairs the split before training, so its numbers are the honest
   ones — and they may be lower. That is the audit working, not a regression.

2. **Preprocessing everywhere means the app must do it too.** The headline accuracy assumes
   `preprocess_image` runs at inference. If the Flask app skips it, you get the OFF row
   from section 10b instead. `run_config.json` records `preprocess_splits` so the app can
   check.

3. **Tune the gates on train only.** `BLUR_VARIANCE_MIN` and the brightness band are fitted
   by eye in section 4b against the training split. Re-tuning them after seeing test
   results turns a per-image transform into a fitted one, and then it really would leak.

4. **The new leaf mask assumes a photographable background.** It samples the image border.
   On your dataset — leaves on a flat surface — that is solid. On a field photo the border
   is more leaf, the chroma MAD rises, and the code falls back to the old excess-green mask
   and sets `uniform_background = False`. Filter on that flag rather than trusting the
   number.

5. **`LEAF_CLOSE_RATIO` is deliberately small.** Raising it seals awkward notches. It also
   fills the gaps between the lobes of a grape leaf, and every background pixel it traps
   becomes "inside the leaf and not green" — a lesion. If a leaf needs more closing than
   0.02, look at the image rather than the constant.

6. **Severity bands are uncalibrated.** The rebuilt mask counts necrotic tissue the old one
   deleted, so the same leaf reads higher and 5/15/35 is an estimate. Move them against
   your own histogram in section 6 before the words Mild/Moderate/High reach a farmer.

7. **Boxes are pseudo labels.** No mAP, no box IoU. Report classification metrics, and
   treat severity % as indicative — section 5a validates the *measurement*, not the
   clinical meaning of the number.

8. **The advisory layer lives in another notebook.** This one stops at class, severity and
   masks. Everything it exports — `metrics.json`, `predictions.csv`, `severity.py`,
   `preprocessing.py`, `best_cnn.pt` — is what the RAG notebook and the app consume.